# Generate AI reviews with OPEN models (Route B, non-GPT families)
Adds cross-**family** generator diversity (the proven lever for generalization) on top
of the OpenAI/GPT reviews: **Qwen2.5-7B**, **Llama-3-8B**, **Phi-3-mini** — all
ungated (no HF token needed). Topic-matched to real products, with varied persona /
length / style / rating (incl. short & casual), same design as `generate_ai_reviews.py`.

### Instructions
1. **Runtime -> Change runtime type -> A100 GPU** (7-8B models in fp16; T4 may OOM — set `LOAD_4BIT=True`).
2. Run all cells (takes a while — generation is slower than the API).
3. Download `ai_generated_reviews_open.csv` and put it in your project's `data/` folder.
4. Tell the assistant; it will add these as CG sources, rebuild, re-measure LOGO, then retrain.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os, csv, json, random, gc, urllib.request
import torch
from transformers import pipeline, BitsAndBytesConfig

SEED = 42
random.seed(SEED)

# (model_id, short family name).  All ungated.
MODELS = [
    ("Qwen/Qwen2.5-7B-Instruct",              "qwen"),
    ("NousResearch/Meta-Llama-3-8B-Instruct", "llama3"),
    ("microsoft/Phi-3-mini-4k-instruct",      "phi3"),
]
PER_MODEL = 3000          # reviews per model (~9k total); lower to save Colab time
BATCH     = 16
LOAD_4BIT = False         # set True on a small GPU (T4) to fit; slower

PERSONAS = ["a busy parent","a college student","a retiree on a fixed budget","a professional chef",
            "a skeptical first-time buyer","a loyal repeat customer","a fitness enthusiast","a bargain hunter",
            "a gift-giver","a picky eater","someone who reads every ingredient label","a small-business owner"]
LENGTHS  = ["in just one short sentence","in one or two sentences","in a short paragraph",
            "in a detailed paragraph or two"]
STYLES   = ["polished and well-written","casual and conversational",
            "quick and informal, all lowercase with a couple of typos like a phone review",
            "enthusiastic and full of energy","measured, balanced and a little critical","blunt and to the point"]
STARS    = [5,5,5,5,4,4,4,3,2,1,1]
CATS     = ["Grocery_and_Gourmet_Food","Health_and_Household","Electronics","Home_and_Kitchen",
            "Office_Products","Beauty_and_Personal_Care","Sports_and_Outdoors","Pet_Supplies"]
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Topic-matched product seeds: stream real product (title + desc) from Amazon-2023 meta
def hf_url(fn): return f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/{fn}"
seeds = []
for cat in CATS:
    got = 0
    try:
        req = urllib.request.Request(hf_url(f"raw/meta_categories/meta_{cat}.jsonl"), headers={"User-Agent":"x"})
        with urllib.request.urlopen(req) as resp:
            for raw in resp:
                try: row = json.loads(raw)
                except Exception: continue
                title = (row.get("title") or "").strip()
                if len(title) < 5: continue
                desc = ""
                d = row.get("description") or []
                if isinstance(d, list) and d: desc = " ".join(str(x) for x in d)
                feats = row.get("features") or []
                if isinstance(feats, list) and feats: desc = (desc+" "+" ".join(str(x) for x in feats)).strip()
                seeds.append((title[:160], desc[:400])); got += 1
                if got >= 1500: break
    except Exception as e:
        print("seed err", cat, e)
    print(f"seeds[{cat}]: {got}")
random.shuffle(seeds)
print("total seeds:", len(seeds))

In [ ]:
def build_prompt(title, desc):
    persona = random.choice(PERSONAS); length = random.choice(LENGTHS)
    style = random.choice(STYLES); stars = random.choice(STARS)
    ctx = f"Product: {title}." + (f" Details: {desc}" if desc else "")
    user = (f"Write a realistic online customer review {length}, as if you are {persona}, "
            f"giving {stars} out of 5 stars. Write it {style}. Sound like a genuine human shopper - "
            f"mention specifics, do not be generic, and do NOT include the star rating, a title, or any "
            f"preamble. Just output the review text.\n\n{ctx}")
    return user, persona, style, stars

In [ ]:
OUT = "ai_generated_reviews_open.csv"
new = not os.path.exists(OUT)
f = open(OUT, "a", newline="", encoding="utf-8"); w = csv.writer(f)
if new: w.writerow(["text","domain","source","model_id","rating","persona","style"])
written = 0

for model_id, fam in MODELS:
    print("\n=== loading", model_id, "===")
    kw = dict(torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    if LOAD_4BIT:
        kw = dict(model_kwargs={"quantization_config": BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},
                  device_map="auto", trust_remote_code=True)
    try:
        pipe = pipeline("text-generation", model=model_id, **kw)
    except Exception as e:
        print("  load failed, skipping:", e); continue
    tok = pipe.tokenizer
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"

    prompts, meta = [], []
    for _ in range(PER_MODEL):
        title, desc = random.choice(seeds)
        user, persona, style, stars = build_prompt(title, desc)
        prompts.append([{"role":"user","content":user}]); meta.append((stars,persona,style))

    for i in range(0, len(prompts), BATCH):
        batch = prompts[i:i+BATCH]
        try:
            outs = pipe(batch, max_new_tokens=256, do_sample=True, temperature=0.85, top_p=0.95,
                        batch_size=len(batch), pad_token_id=tok.pad_token_id)
        except Exception as e:
            print("  gen error:", e); continue
        for j, o in enumerate(outs):
            rec = o[0] if isinstance(o, list) else o
            gen = rec["generated_text"]
            text = gen[-1]["content"] if isinstance(gen, list) else str(gen)
            text = text.strip().replace("\n", " ")
            stars, persona, style = meta[i+j]
            if len(text.split()) >= 4:
                w.writerow([text, "products", fam, model_id, stars, persona, style]); written += 1
        f.flush()
        if (i // BATCH) % 10 == 0: print(f"  {fam}: {i+len(batch)}/{PER_MODEL}  (total {written})")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    print(fam, "done")
f.close()
print("\nTOTAL written:", written)

In [ ]:
from google.colab import files
files.download("ai_generated_reviews_open.csv")

## Next steps
Put `ai_generated_reviews_open.csv` in your project's `data/` folder and tell the
assistant. It will add `qwen` / `llama3` / `phi3` as new CG sources, rebuild
`fake_reviews_broadened.csv`, re-run `evaluate_cgor_generalization.py` (expect mean
LOGO recall to rise again), and then retrain the final models once on the complete set.